# 數位控制系統第五章：閉迴路系統（教學版 Notebook）

本 Notebook 是 `chp5.md` 教材的教學版，額外補充：

- 每個第一次出現的 MATLAB / Octave 函數（`feedback`, `minreal`, `pole`, `tfdata`, ...）的逐步解說
- 複數極點的正確列印、`feedback` 的符號慣例等初學者容易卡住的地方
- 公式 → 程式碼的逐項對照（每段程式都標註對應的教材式號與範例編號）
- 六個可直接執行的實驗，親手驗證本章每一條公式

建議搭配 `chp5.md`（完整理論、推導與符號定義）與 `chp5.m`（精簡可執行版）一起閱讀。

> **本章要做的事**：第 4 章建立了**開迴路**的 $C(z)=G(z)E(z)$。本章把輸出接回輸入端，得到**閉迴路**的結果——這才是所有實際控制系統的形態。

---


## 🔧 環境設定

> **需要 Control System Toolbox**（Octave 為 `control` 套件）。本章大量使用 `feedback`、`c2d`、`tf`、`ss`。


In [ ]:
%plot --format svg

if exist('OCTAVE_VERSION', 'builtin')
    warning('off', 'Octave:gnuplot-graphics');
    warning('off', 'Octave:fltk-graphics');
    graphics_toolkit('gnuplot');
    pkg load control;
end
clear; clc;

set(0, 'DefaultTextFontName', 'Microsoft JhengHei');
set(0, 'DefaultAxesFontName', 'Microsoft JhengHei');

T = 0.1;                  % 取樣週期 (s)
printf('取樣週期 T = %g s\n', T);

## 📖 全章符號總表

| 符號 | 型別 | 意義 |
|---|---|---|
| $R(z)$ | 複變函數 | 參考輸入 |
| $C(z)$ | 複變函數 | 系統輸出 |
| $G(z)$ | 複變函數 | 前向路徑（**含資料保持器**） |
| $H(s)$ | 複變函數 | 回授路徑（感測器） |
| $D(z)$ | 複變函數 | 數位控制器 |
| $\overline{GH}(z)$ | 複變函數 | $\mathcal{Z}[G(s)H(s)]$，**先在 $s$ 域相乘再取 z 轉換** |
| $\mathbf{v}(k)$ | $n\times1$ | 離散狀態向量 |
| $A_1,B_1,B_2$ | 矩陣 | **尚未關迴路**時的狀態矩陣 |
| $C_1,D_1$ | 矩陣 | **濾波器**的輸出方程式矩陣 |
| $C$ | 矩陣 | **受控體**的輸出矩陣 |
| $m(k)$ | 純量 | 濾波器輸出 = 受控體輸入 |
| $e(k)$ | 純量 | 誤差 $=u(k)-y(k)$ |

## 📖 本章會用到的 Octave 語法小抄

| 語法 | 意義 | 注意事項 |
|---|---|---|
| `feedback(sys1, sys2)` | 負回授 $\dfrac{sys_1}{1+sys_1sys_2}$ | **預設是負回授**；正回授要寫 `feedback(s1,s2,+1)` |
| `feedback(sys, 1)` | 單位負回授 $\dfrac{sys}{1+sys}$ | 最常用 |
| `pole(sys)` | 系統極點 | **可能是複數** |
| `minreal(sys)` | 最小實現（消掉共同因子） | 手動組合出來的系統常需要 |
| `tfdata(sys,'v')` | 取出分子分母係數向量 | `'v'` = vector |
| `ssdata(sys)` | 取出 $A,B,C,D$ | |
| `abs(p)`, `real(p)`, `imag(p)` | 複數的絕對值、實部、虛部 | 判斷穩定看 `abs(p)<1` |
| `merge(cond, a, b)` | Octave 的三元運算 | MATLAB 沒有，要改用 `if` |

> **⚠️ 列印複數極點的陷阱**
>
> ```matlab
> fprintf('%.5f ', pole(sys))     % 只印出實部，虛部被吃掉！
> ```
>
> 若極點是 $0.936\pm j0.286$，上面只會印出兩個 `0.93609`，讓你誤以為是重根。**正確寫法**：
>
> ```matlab
> pp = pole(sys);
> fprintf('%9.5f%+8.5fj  ', [real(pp)'; imag(pp)']);
> ```
>
> `fprintf` 遇到 $2\times N$ 矩陣會**逐欄取值**，剛好把 (實部, 虛部) 配成一對。

---


## 二、標準閉迴路輸出函數 (5.2，式 5-11)

### 推導的關鍵一步

```text
R(s) ──►(+)──► E(s) ──/──► E*(s) ──► G(s) ──┬──► C(s)
         ▲(-)          T                     │
         └────────────── H(s) ◄──────────────┘
```

$$C(s)=G(s)E^*(s),\qquad E(s)=R(s)-H(s)C(s)$$

代入後 $E(s)=R(s)-G(s)H(s)E^*(s)$。**對整個式子取星號轉換**（$E^*(s)$ 只以 $e^{Ts}$ 形式出現，可以被提出來）：

$$E^*(s)=R^*(s)-\overline{GH}^*(s)E^*(s)\ \Longrightarrow\ E^*(s)=\frac{R^*(s)}{1+\overline{GH}^*(s)}$$

$$\boxed{C(z)=\frac{G(z)R(z)}{1+\overline{GH}(z)}}$$

> **⚠️ 分母是 $\overline{GH}(z)$ 不是 $G(z)H(z)$**——橫線代表**先在 $s$ 域相乘再取 z 轉換**。這是第 4 章式 (4-19) 在閉迴路的延續，也是本章最容易寫錯的地方。

### 一個重要的實作假設（教材明訂）

> 因為 $E^*(s)$ 是一連串脈衝函數，**必須假設 $G(s)$ 是透過零階保持器接收這些脈衝的**。
>
> **只要對一個拉氏轉移函數套用星號轉換，就假設它已含有一個零階保持器。**

### `feedback` 在做什麼

`feedback(G, H)` 計算 $\dfrac{G}{1+GH}$。單位回授時寫 `feedback(G, 1)`。

下一格順便手動用 `Gz/(1+Gz)` 算一次，驗證兩者相同。**手動組出來的系統會有多餘的共同因子，要用 `minreal` 消掉**。


In [ ]:
%% 實驗 1：標準閉迴路（式 5-11，單位回授）
Gp = tf(1, [1 1]);          % 受控體 Gp(s) = 1/(s+1)
Gz = c2d(Gp, T, 'zoh');     % G(z)，含零階保持器
Tz = feedback(Gz, 1);       % 單位負回授 G(z)/(1+G(z))

[ng, dg] = tfdata(Gz, 'v');
[nt, dt] = tfdata(Tz, 'v');
printf('開迴路 G(z) = %.6f / (z %+.6f)\n', ng(2), dg(2));
printf('閉迴路 T(z) = %.6f / (z %+.6f)\n', nt(2), dt(2));

% 手動驗證 feedback 的定義
Tz_manual = Gz / (1 + Gz);
printf('\n手動 G/(1+G) 與 feedback 的極點差 = %.2e\n', ...
       max(abs(sort(pole(Tz)) - sort(pole(minreal(Tz_manual))))));
printf('閉迴路 dc 增益 = %.6f\n', dcgain(Tz));
printf('開迴路極點 %.5f -> 閉迴路極點 %.5f （回授把極點往原點拉）\n', ...
       -dg(2), -dt(2));

### 結果解讀

**回授把極點從 $0.9048$ 拉到 $0.8097$**——更靠近原點代表**衰減更快、反應更快**（第 3、4 章與 Ackermann 範例都看過這個現象）。

但代價是 **dc 增益從 1 掉到 0.5**——閉迴路產生了穩態誤差。這正是第 6 章第 6.5 節「穩態精確度」要處理的問題。

> **這就是控制設計永恆的取捨**：回授讓系統變快變穩，但也帶來穩態誤差；要消除穩態誤差就得加積分器，而積分器又會讓系統趨向不穩定。

---


## 三、⚠️ $\overline{GH}(z)$ 與 $G(z)H(z)$ 的差別

### 什麼時候用哪一個

這完全取決於**實體上感測器與受控體之間有沒有取樣器**：

| 實體架構 | 分母該用 | 程式 |
|---|---|---|
| $G$ 與 $H$ 之間**沒有**取樣器（常見） | $\overline{GH}(z)=\mathcal{Z}[G(s)H(s)]$ | `c2d(Gs*Hs, T, 'zoh')` |
| $G$ 與 $H$ 之間**有**取樣器 | $G(z)H(z)$ | `c2d(Gs,T,'zoh')*c2d(Hs,T,'zoh')` |

> **⚠️ `feedback(Gz, Hz)` 算的是 $\dfrac{G(z)}{1+G(z)H(z)}$**——也就是**第二種**情況。
>
> 若你的系統屬於第一種（感測器直接接在受控體輸出、中間沒有 A/D），**不能直接用 `feedback(Gz, Hz)`**，必須自己組 $\dfrac{G(z)}{1+\overline{GH}(z)}$。

下一格用數值展示兩者差多少。


In [ ]:
%% 實驗 2：GHbar(z) vs G(z)H(z)（式 5-11 分母的陷阱）
Hs = tf(2, [1 5]);                        % 感測器 H(s) = 2/(s+5)

GH_bar = c2d(Gp*Hs, T, 'zoh');            % 正確（中間無取樣器）：先在 s 域相乘
GH_sep = Gz * c2d(Hs, T, 'zoh');          % 中間有取樣器：各自離散化後相乘

[n1, ~] = tfdata(GH_bar, 'v');
[n2, ~] = tfdata(GH_sep, 'v');
printf('GHbar(z) 分子 = [%.6g %.6g %.6g]\n', n1);
printf('G(z)H(z) 分子 = [%.6g %.6g %.6g]\n', n2);
printf('兩者 dcgain：%.6f  vs  %.6f （穩態相同）\n\n', dcgain(GH_bar), dcgain(GH_sep));

Tz_sep = feedback(Gz, c2d(Hs, T, 'zoh')); % feedback 用的是 G(z)H(z)
Tz_bar = minreal(Gz / (1 + GH_bar));      % 自己組 G(z)/(1+GHbar(z))

printf('兩種閉迴路的極點：\n');
pp = sort(pole(Tz_sep)); printf('  用 G(z)H(z) : '); printf('%9.5f%+8.5fj  ', [real(pp)'; imag(pp)']); printf('\n');
pp = sort(pole(Tz_bar)); printf('  用 GHbar(z) : '); printf('%9.5f%+8.5fj  ', [real(pp)'; imag(pp)']); printf('\n');

figure('Position', [50 50 800 380]);
[ya, ta] = step(Tz_sep, 0:T:2);
[yb, ~]  = step(Tz_bar, 0:T:2);
stairs(ta, ya, 'b-', 'LineWidth', 2); hold on;
stairs(ta, yb, 'r--', 'LineWidth', 2); grid on;
title('分母用 G(z)H(z) vs GHbar(z)');
xlabel('時間 t (秒)'); ylabel('c(kT)');
legend('用 G(z)H(z)（中間有取樣器）', '用 GHbar(z)（中間無取樣器）', 'Location', 'southeast');

### 結果解讀

兩者的**極點明顯不同**（$0.670,0.841$ vs $0.655,0.848$），階躍響應也不同。

**這不是誰對誰錯，而是對應兩種不同的實體架構**——你必須看方塊圖決定用哪一個。

> **實務判斷法**：問「感測器的訊號有沒有經過 A/D 轉換器？」
> - **有** → 中間有取樣器 → 用 $G(z)H(z)$
> - **沒有**（類比感測器直接接回加法點） → 用 $\overline{GH}(z)$

---


## 四、含數位控制器的閉迴路 (例 5.1)

### 四步驟程序

第 5.2 節的做法有一個陷阱：**星號化的順序會影響能不能解出來**。若先把 $E(s)=R(s)-H(s)C(s)$ 星號化再代入，會得到

$$C^*(s)=G^*(s)R^*(s)-G^*(s)\overline{HC}^*(s)$$

而 $C^*(s)$ **無法從 $\overline{HC}^*(s)$ 中提出來**——解不出來。

**第 5.3 節的四步驟程序系統化地避開這個問題**：

| 步驟 | 內容 |
|---|---|
| 1 | 建立原始訊號流圖（**省略取樣器**，但把取樣器輸出以星號形式保留） |
| 2 | 對每個取樣器輸入指定變數，輸出就是該變數加星號 |
| 3 | **把每個取樣器輸出視為輸入（來源節點）**，用它們與系統輸入表示所有取樣器輸入與系統輸出 |
| 4 | 對這些方程式取星號轉換，再求解 |

> **💡 為什麼這樣就不會踩到陷阱？** 因為步驟 3 強制你先把所有東西寫成「取樣器輸出」的函數。星號化時被提出的因子必定是取樣器輸出（本來就是星號形式），**不可能發生訊號失去因子地位的問題**。

### 例 5.1 的結果

$$\boxed{C(z)=\frac{D(z)G(z)}{1+D(z)\overline{GH}(z)}R(z)}$$

**這是數位控制系統最標準的閉迴路轉移函數**，第 8、9 章的控制器設計會反覆用到。單位回授（$H=1$）時就是 `feedback(Dz*Gz, 1)`。


In [ ]:
%% 實驗 3：含數位控制器的閉迴路（例 5.1）
Dz  = tf([0.9 -0.8], [1 -0.9], T);   % D(z) = (0.9z-0.8)/(z-0.9)
Gp3 = tf(10, [1 1 0]);               % Gp(s) = 10/(s(s+1))，含純積分器
Gz3 = c2d(Gp3, T, 'zoh');
Tz3 = feedback(Dz*Gz3, 1);           % C(z)/R(z) = D(z)G(z)/(1+D(z)G(z))

[n3, d3] = tfdata(Tz3, 'v');
printf('T(z) 分子 = [%.6g %.6g %.6g %.6g]\n', n3);
printf('T(z) 分母 = [%.6g %.6g %.6g %.6g]\n\n', d3);

pp = sort(pole(Tz3));
printf('極點   : '); printf('%9.5f%+8.5fj  ', [real(pp)'; imag(pp)']); printf('\n');
printf('絕對值 : '); printf('%9.5f          ', abs(pp)'); printf('\n');
printf('全部在單位圓內？ 最大絕對值 = %.5f\n', max(abs(pp)));
printf('開迴路 Gp(s) 含純積分器 -> 開迴路 dcgain 無限大；閉迴路 dcgain = %.6f\n', dcgain(Tz3));

### 結果解讀

**三個極點：一個實根 $0.889$，一對共軛複根 $0.936\pm j0.286$**（絕對值 $0.979$）。

**注意這裡的複數極點**——若用 `fprintf('%.5f ', pole(Tz3))`，你只會看到 `0.88912 0.93609 0.93609`，**誤以為是重根**。虛部 $\pm0.286$ 被完全吃掉了。這就是上面語法小抄警告的陷阱。

**閉迴路 dc 增益 = 1.0**——因為開迴路含**純積分器**（$G_p(s)=\dfrac{10}{s(s+1)}$ 在 $s=0$ 有極點），開迴路 dc 增益無限大，閉迴路就沒有穩態誤差。**這就是第 6 章第 6.5 節「系統型式」的概念**：型式 1 以上的系統，對步階輸入的穩態誤差為零。

---


## 五、閉迴路狀態變數模型：把迴路關起來 (5.4，式 5-38)

### 想法

把**數位濾波器**與**受控體**分開處理，各自寫狀態方程式，**然後用一條公式把回授迴路「關」起來**。

**三組方程式**：

$$\mathbf{v}(k+1)=A_1\mathbf{v}(k)+B_1m(k)+B_2e(k)\qquad(m\text{ 與 }e\text{ 都當輸入})$$

$$m(k)=C_1\mathbf{v}(k)+D_1e(k)\qquad(\text{濾波器輸出})$$

$$y(k)=C\mathbf{v}(k)\qquad(\text{受控體輸出})$$

**回授**：$e(k)=u(k)-y(k)=u(k)-C\mathbf{v}(k)$

**消去 $m(k)$ 與 $e(k)$** 後得到本章核心公式：

$$\boxed{A=A_1+B_1C_1+(-B_2-B_1D_1)C,\qquad B=B_1D_1+B_2}$$

> **💡 這條公式在做什麼**：$A_1,B_1,B_2,C_1,D_1,C$ 全部是**迴路還沒接起來**時各方塊的矩陣。這條公式**用一次矩陣運算就把回授迴路關起來**——不需要畫模擬圖、不需要查 z 轉換表。

### 例 5.7 的組成

| 部分 | 內容 |
|---|---|
| 受控體（2 階） | 第 4 章例 4.13：$A=\begin{bmatrix}1&0.0952\cr0&0.905\end{bmatrix}$，$B=\begin{bmatrix}0.0484\cr0.952\end{bmatrix}$，$C=[1\ 0]$ |
| 濾波器（1 階） | $D(z)=\dfrac{0.9z-0.8}{z-0.9}$：$v_3(k+1)=0.9v_3(k)+e(k)$，$m(k)=0.01v_3(k)+0.9e(k)$ |

合併成三階系統後，對照式 (5-32)~(5-35) 讀出：

$$A_1=\begin{bmatrix}1&0.0952&0\cr0&0.905&0\cr0&0&0.9\end{bmatrix},\quad
B_1=\begin{bmatrix}0.0484\cr0.952\cr0\end{bmatrix},\quad
B_2=\begin{bmatrix}0\cr0\cr1\end{bmatrix}$$

$$C_1=[0\ \ 0\ \ 0.01],\quad D_1=0.9,\quad C=[1\ \ 0\ \ 0]$$

### 濾波器的狀態方程式怎麼來的

$D(z)=\dfrac{0.9z-0.8}{z-0.9}$ 用**直接形式**實現：先做長除法把它拆成「常數 + 真分式」

$$\frac{0.9z-0.8}{z-0.9}=0.9+\frac{0.9\times0.9-0.8}{z-0.9}=0.9+\frac{0.01}{z-0.9}$$

- **常數項 $0.9$** → 直接傳遞項 $D_1=0.9$
- **真分式 $\dfrac{0.01}{z-0.9}$** → 一個狀態：$v_3(k+1)=0.9v_3(k)+e(k)$，貢獻 $0.01v_3(k)$ 到輸出

這就是 $m(k)=0.01v_3(k)+0.9e(k)$ 的來歷（教材寫成 $0.81-0.8=0.01$，同一件事）。


In [ ]:
%% 實驗 4：關迴路公式（式 5-38，例 5.7）
A1 = [1 0.0952 0; 0 0.905 0; 0 0 0.9];   % 受控體(2階) + 濾波器(1階)，尚未關迴路
B1 = [0.0484; 0.952; 0];                 % m(k) 的輸入矩陣
B2 = [0; 0; 1];                          % e(k) 的輸入矩陣
C1 = [0 0 0.01];                         % 濾波器輸出 m = C1*v + D1*e
D1 = 0.9;
C  = [1 0 0];                            % 受控體輸出 y = C*v

A = A1 + B1*C1 + (-B2 - D1*B1)*C;        % 式 (5-38)
B = D1*B1 + B2;

printf('閉迴路 A =\n'); disp(A);
printf('閉迴路 B = [%.5f  %.5f  %.5f]''\n\n', B);
printf('教材 A = [0.9564 0.0952 0.0005; -0.8568 0.9050 0.0095; -1 0 0.9]\n');
printf('教材 B = [0.0436; 0.8568; 1.0000]\n');
printf('=> 完全吻合\n');

% 順便驗證濾波器的長除法拆解
printf('\n濾波器 D(z) = (0.9z-0.8)/(z-0.9) 的直接形式拆解：\n');
printf('  常數項 D1      = %.4f\n', 0.9);
printf('  真分式分子     = 0.9*0.9 - 0.8 = %.4f  -> C1 的非零元素\n', 0.9*0.9-0.8);

### 結果解讀

式 (5-38) 算出的 $A$、$B$ 與教材例 5.7 **完全吻合**。

**這個做法的價值**：整個過程只有矩陣加乘，**沒有 z 轉換、沒有查表、沒有部分分式**。對高階系統這是唯一實用的做法。

---


## 六、兩種方法交叉驗證 (例 5.7)

### 教材的兩段程式

教材先用**狀態空間法**（式 5-38）算出 $A,B,C,D$，再轉成轉移函數；然後用**轉移函數法**（`feedback`）獨立算一次，交叉比對。

### ⚠️ 教材程式的相容性問題

```matlab
[num,den] = ss2tf(A,B,C,D,1);    % <- Octave 沒有 ss2tf
```

**修正版**（數學內容完全相同）：

```matlab
Tz = tf(ss(A, B, C, 0, T));
```

而 `feedback(Dz*Gz, 1)` 那一段**在 Octave 可以直接執行**，不需要修改。

### 教材的結果

$$T(z)=\frac{0.04356z^2+0.003426z-0.03746}{z^3-2.761z^2+2.623z-0.852}\qquad\text{（狀態空間法）}$$

$$T_{\text{check}}(z)=\frac{0.04354z^2+0.00341z-0.03743}{z^3-2.761z^2+2.623z-0.8518}\qquad\text{（轉移函數法）}$$

**兩者有微小差異**（$0.04356$ vs $0.04354$）——下一格與實驗 6 會解釋原因。


In [ ]:
%% 實驗 5：狀態空間法 vs 轉移函數法（例 5.7）
% 教材原始寫法 [num,den]=ss2tf(A,B,C,D,1) 在 Octave 無法執行
Tz_ss = tf(ss(A, B, C, 0, T));       % 修正版：用 tf(ss(...)) 取代 ss2tf
Tz_tf = feedback(Dz*Gz3, 1);         % 轉移函數法（教材的 Tz_check）

[na, da] = tfdata(Tz_ss, 'v');
[nb, db] = tfdata(Tz_tf, 'v');
printf('[狀態空間法] 分子 = [%.6g %.6g %.6g %.6g]\n', na);
printf('             分母 = [%.6g %.6g %.6g %.6g]\n', da);
printf('教材 Tz = (0.04356 z^2 + 0.003426 z - 0.03746)/(z^3 - 2.761 z^2 + 2.623 z - 0.852)\n\n');
printf('[轉移函數法] 分子 = [%.6g %.6g %.6g %.6g]\n', nb);
printf('             分母 = [%.6g %.6g %.6g %.6g]\n', db);
printf('教材 Tz_check = (0.04354 z^2 + 0.00341 z - 0.03743)/(z^3 - 2.761 z^2 + 2.623 z - 0.8518)\n\n');

printf('兩種方法的極點：\n');
pp = sort(pole(Tz_ss)); printf('  狀態空間法 : '); printf('%9.5f%+8.5fj  ', [real(pp)'; imag(pp)']); printf('\n');
pp = sort(pole(Tz_tf)); printf('  轉移函數法 : '); printf('%9.5f%+8.5fj  ', [real(pp)'; imag(pp)']); printf('\n');

figure('Position', [50 50 800 380]);
[y1, t1] = step(Tz_ss, 0:T:4);
[y2, ~]  = step(Tz_tf, 0:T:4);
stairs(t1, y1, 'b-', 'LineWidth', 2); hold on;
stairs(t1, y2, 'r--', 'LineWidth', 1.5); grid on;
title('例 5.7：狀態空間法 vs 轉移函數法');
xlabel('時間 t (秒)'); ylabel('y(kT)');
legend('狀態空間法（式 5-38）', '轉移函數法（feedback）', 'Location', 'southeast');

### 結果解讀

兩條階躍響應**幾乎完全重疊**，極點也只差在小數第 4~5 位。

**兩種方法都與教材的輸出完全吻合**——包括那個微小的差異。下一格解釋為什麼會有差異。

---


## 七、為什麼兩種方法有微小差異

### 不是誰算錯了，是輸入資料的精度不同

| 路徑 | 使用的數值 |
|---|---|
| 狀態空間法 | 用**教材四捨五入過的** $A_1$、$B_1$（$0.0952$、$0.905$、$0.0484$、$0.952$） |
| 轉移函數法 | `c2d` 內部用**全精度**值（$0.0951626$、$0.9048374$、$0.0483742$、$0.9516258$） |

下一格用**全精度值重做式 (5-38)**，看差異會不會消失。


In [ ]:
%% 實驗 6：四捨五入誤差的傳播
[Ad, Bd] = ssdata(c2d(ss([0 1;0 -1],[0;10],[1 0],0), T, 'zoh'));
printf('教材四捨五入值 : A(1,2)=%.7f  A(2,2)=%.7f  B(1)=%.7f  B(2)=%.7f\n', ...
       0.0952, 0.905, 0.0484, 0.952);
printf('c2d 全精度值   : A(1,2)=%.7f  A(2,2)=%.7f  B(1)=%.7f  B(2)=%.7f\n\n', ...
       Ad(1,2), Ad(2,2), Bd(1), Bd(2));

% 用全精度值重做式 (5-38)
A1e = [Ad(1,1) Ad(1,2) 0; 0 Ad(2,2) 0; 0 0 0.9];
B1e = [Bd(1); Bd(2); 0];
Ae  = A1e + B1e*C1 + (-B2 - D1*B1e)*C;
Be  = D1*B1e + B2;
Tz_exact = tf(ss(Ae, Be, C, 0, T));
[ne, ~] = tfdata(Tz_exact, 'v');

printf('與轉移函數法的差異（分子首項）：\n');
printf('  用教材四捨五入值 : %.2e\n', abs(na(2)-nb(2)));
printf('  用全精度值       : %.2e\n', abs(ne(2)-nb(2)));
printf('\n=> 差了 %d 個數量級，證明差異純粹來自輸入資料的精度，\n', ...
       round(log10(abs(na(2)-nb(2))/abs(ne(2)-nb(2)))));
printf('   不是式 (5-38) 或 feedback 有誤。\n');

### 結果解讀

```text
用教材四捨五入值 : 2.32e-05
用全精度值       : 2.08e-17
```

**差了 12 個數量級。** 用全精度值時，兩種方法的結果精確到浮點極限——**證明式 (5-38) 與 `feedback` 在數學上完全等價**，教材那個 $0.04356$ vs $0.04354$ 的差異純粹來自**手算時的四捨五入**。

> **💡 這是很好的一課**：教科書為了排版把中間結果印成 3~4 位有效數字，你若照著抄進程式，誤差會一路傳播到最終結果。**做數值驗證時，中間值一律讓程式自己算，不要抄書上四捨五入過的數字。**

---


## 八、本章總結

### 公式速查表

| 式號 | 公式 | 用途 | 對應實驗 |
|---|---|---|---|
| (5-1) | $C(z)=G_1(z)G_2(z)E(z)$ | 中間**有**取樣器 | — |
| (5-2) | $C(z)=\overline{G_1G_2}(z)E(z)$ | 中間**沒有**取樣器 | 2 |
| (5-11) | $C(z)=\dfrac{G(z)R(z)}{1+\overline{GH}(z)}$ | **標準閉迴路輸出** | 1, 2 |
| (5-18) | $C(z)=\dfrac{\overline{GR}(z)}{1+\overline{GH}(z)}$ | 輸入未經取樣（無轉移函數） | — |
| 例 5.1 | $C(z)=\dfrac{D(z)G(z)}{1+D(z)\overline{GH}(z)}R(z)$ | **數位控制標準式** | 3 |
| (5-38) | $A=A_1+B_1C_1+(-B_2-B_1D_1)C$，$B=B_1D_1+B_2$ | **關迴路公式** | 4, 5, 6 |

### MATLAB / Octave 常見錯誤

| 錯誤 | 症狀 | 正確做法 |
|---|---|---|
| `fprintf('%.5f', pole(sys))` | **虛部被吃掉**，複數極點看起來像重根 | `fprintf('%9.5f%+8.5fj ', [real(p)'; imag(p)'])` |
| 分母用 $G(z)H(z)$ 但實體無取樣器 | 極點與響應都錯 | 用 `c2d(Gs*Hs,T,'zoh')` 組 $\overline{GH}(z)$ |
| 用 `ss2tf` | Octave 沒有這個函數 | `tf(ss(A,B,C,D,T))` |
| 手動組系統後不用 `minreal` | 出現多餘的極零點對 | `minreal(sys)` |
| 抄教科書四捨五入過的中間值 | 最終結果誤差放大 | 讓程式自己算全精度值 |
| 忘記 `feedback` 預設是負回授 | 符號錯誤 | 正回授寫 `feedback(s1,s2,+1)` |

### 承先啟後

本章完成了閉迴路離散系統的兩套分析工具：

1. **轉移函數路線**：$C(z)=\dfrac{D(z)G(z)}{1+D(z)\overline{GH}(z)}R(z)$，配合訊號流圖四步驟程序
2. **狀態空間路線**：式 (5-38) 的關迴路公式，保留物理狀態

**第 6 章**會用這些結果分析**時間響應特性**——閉迴路極點落在 $z$ 平面哪裡，就對應什麼樣的暫態行為；以及**穩態精確度**——系統型式如何決定穩態誤差。

**第 7 章**討論穩定度（極點是否都在單位圓內），**第 8、9 章**則是設計 $D(z)$ 讓極點落在你要的地方——那正是 [`course/ackermann`](../ackermann/full-Ackermann-formula-example.md) 示範的極點安置法。
